# Phase 7 — Model Evaluation

This phase evaluates the saved Phase 6 estimator on the untouched grouped holdout. The exact same group split and target construction are recreated. We compare development and holdout metrics, review residual patterns, and record limitations. Precision@K and NDCG are not calculated because the project does not contain human relevance labels for search-query/product pairs.

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sys.path.insert(0, "..")
from backend.product_features import product_feature_frame

df = pd.read_csv("electronics_feature_engineered.csv")
# Match Phase 6: an absent review count contributes zero before log1p.
df["no_of_ratings"] = pd.to_numeric(df["no_of_ratings"], errors="coerce").fillna(0).clip(lower=0)
artifact = joblib.load("../backend/model_artifacts/product_score.joblib")
X = product_feature_frame(df)
rating = pd.to_numeric(df.review_rating, errors="coerce")
rating_quality = rating.fillna(artifact["target_rating_fill"]).clip(0, 5) / 5
reviews = pd.to_numeric(df.no_of_ratings, errors="coerce").fillna(0).clip(lower=0)
popularity = np.log1p(reviews) / artifact["target_log_review_max"]
discount = pd.to_numeric(df.discount_percent, errors="coerce").clip(0, 100).fillna(0) / 100
y = 0.45 * rating_quality + 0.35 * popularity + 0.20 * discount
groups = df["name"].fillna("").astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
splitter = GroupShuffleSplit(n_splits=1, test_size=artifact["test_size"], random_state=artifact["random_state"])
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
model = artifact["model"]
print("Selected model:", artifact["model_name"])
print("Feature order matches artifact:", X.columns.tolist() == artifact["feature_columns"])
print("Train/test rows:", len(X_train), len(X_test))

Selected model: Random Forest
Feature order matches artifact: True
Train/test rows: 3989 1021


In [3]:
def score_report(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test,test_pred))
evaluation = pd.DataFrame([
    {"Split": "Train", **score_report(y_train, train_pred)},
    {"Split": "Test", **score_report(y_test, test_pred)},
])
display(evaluation.round(4))

print("Test prediction range:", round(float(test_pred.min()), 4), "to", round(float(test_pred.max()), 4))
print("Test residual mean:", round(float(np.mean(y_test - test_pred)), 5))
print("Test residual standard deviation:", round(float(np.std(y_test - test_pred)), 5))
evaluation.to_csv("model_outputs/final_evaluation.csv", index=False)

ValueError: continuous is not supported

In [4]:
residuals = y_test.to_numpy() - test_pred
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(test_pred, residuals, alpha=0.35, s=18)
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set(xlabel="Predicted proxy score", ylabel="Residual (actual − predicted)", title="Holdout residuals")
axes[1].hist(residuals, bins=30, color="#2f6f8f", edgecolor="white")
axes[1].set(xlabel="Residual", ylabel="Product count", title="Residual distribution")
plt.tight_layout()
fig.savefig("model_outputs/phase7_residual_diagnostics.png", dpi=180, bbox_inches="tight")
plt.show()

print("Limitations")
print("- The response variable is a constructed score, not a user preference label.")
print("- Product variants with different title text may still occur across the split.")
print("- Specification fields are sparse because titles often omit hardware details.")
print("- Live relevance metrics require query-product judgments that are not available.")

07_model_evaluation.ipynb:cell3:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Limitations
- The response variable is a constructed score, not a user preference label.
- Product variants with different title text may still occur across the split.
- Specification fields are sparse because titles often omit hardware details.
- Live relevance metrics require query-product judgments that are not available.


## Evaluation conclusion

Use this model only as a supporting proxy score. The live recommendation score must preserve explicit category and budget constraints and can use requirement-specific matches for RAM, storage, display, GPU, or use case. A future evaluation should label a representative set of query/product pairs and compute ranking metrics such as Precision@5 and NDCG@5.